# Quantized SPD-RAG pilot — Kaggle runner

**Research question.** Does hierarchical multi-agent RAG stay robust under aggressive
weight quantization because each document agent solves a small focused problem, or do
small branch-level failures compound as the number of independently required RAG agents
grows?

Three precisions from **one** source checkpoint: `F16`, `Q8_0`, `Q4_K_M`.

**Before you start**
1. Settings → Accelerator → **GPU** (T4 x2 or P100).
2. Settings → Internet → **On** (needed to clone llama.cpp and download the checkpoint;
   if internet is off, attach them as input datasets and use the `--skip-*` flags below).
3. Attach the MultiHop-RAG files (`MultiHopRAG.json`, `corpus.json`) as an input dataset.

**Stage sizes.** Stage A = 18 predictions. Stage B = 36. Stage C = 18. Maximum 72.
The run stops after Stage A unless its gate passes.


## 0. Repo location and CUDA check


In [ ]:
import os, subprocess, sys, json, pathlib

# Point REPO at wherever you put this repository (an input dataset, a git clone,
# or an unzipped upload). It must be writable, so copy it into /kaggle/working.
SRC  = pathlib.Path('/kaggle/input/quantized-spd-rag-pilot')   # <-- edit if needed
REPO = pathlib.Path('/kaggle/working/quantized-spd-rag-pilot')

if not REPO.exists():
    if SRC.exists():
        subprocess.run(['cp','-r',str(SRC),str(REPO)], check=True)
    else:
        raise SystemExit(f'Repository not found. Upload it, then set SRC. Looked in {SRC}')
os.chdir(REPO)
sys.path.insert(0, str(REPO/'src'))
print('repo:', REPO)
print(subprocess.run(['nvidia-smi'], capture_output=True, text=True).stdout)


## 1. Dependencies


In [ ]:
!pip install -q pydantic pyyaml psutil sentence-transformers 2>&1 | tail -2
import numpy, pydantic, yaml; print('numpy', numpy.__version__, '| pydantic', pydantic.VERSION)


## 2. Preflight — CUDA, input discovery, GPU offload proof

This writes `results/kaggle_preflight.json`. `--skip-preflight` is only for the very
first pass, before any GGUF exists.


In [ ]:
!python scripts/prepare_kaggle.py --skip-preflight


## 3. Build llama.cpp with CUDA, then make the three GGUFs

One conversion path: HF checkpoint → **one** F16 GGUF → `Q8_0` and `Q4_K_M` quantized
from that exact file. SHA-256, sizes, commands, source revision and the llama.cpp
commit all land in `results/provenance.json`.

Takes roughly 20–40 minutes the first time. Everything is cached, so re-running is cheap.

*No internet?* Attach the checkpoint as an input dataset and add
`--skip-download --hf-dir /kaggle/input/<your-dataset>`.


In [ ]:
!python scripts/prepare_models.py --build-llama-cpp


In [ ]:
# Confirm GPU offload with a real load of the smallest model
!python scripts/prepare_kaggle.py --preflight-precision Q4_K_M


## 4. Dataset → frozen manifest and seed rubric

Deterministic sampling. `data/pilot_manifest.json` is frozen once written.


In [ ]:
!python scripts/prepare_dataset.py --data-dir data --search /kaggle/input


### Review the rubric (recommended before quoting any number)

`data/gold_facts.json` is seeded from the dataset's own evidence facts and every entry
is flagged `needs_manual_review: true`. Reports stay stamped `rubric_reviewed: false`
until you clear them.


In [ ]:
import json
gold = json.load(open('data/gold_facts.json'))
q0 = next(iter(gold['questions'].values()))
print(q0['question'])
print('gold answer:', q0['answer'])
for f in q0['facts']:
    print(' -', f['document_id'], '|', f['text'][:120])


## 5. Indexes — one private index per document, isolation proved before any model runs


In [ ]:
!python scripts/build_indexes.py --stage all
print(open('results/isolation_report.json').read()[:400])


## 6. Stage A — capability calibration (18 predictions)

One precision block at a time: F16, then Q8_0, then Q4_K_M. Question order is
counterbalanced per block and recorded. Resumable — rerun the cell after an
interruption and it picks up where it stopped.


In [ ]:
!python scripts/run_stage.py --stage stage_a --backend llama-server


In [ ]:
!python scripts/score.py --stage stage_a --emit-adjudication
!python scripts/analyze.py --stage stage_a --ni-margin 0.05
!python scripts/package_results.py --stage stage_a


In [ ]:
import json
gate = json.load(open('results/stage_a/gate.json'))
print(json.dumps(gate, indent=2))
print()
print('PROCEED TO STAGE B' if gate['proceed'] else 'STOP — Stage A did not clear its gate')


## 7. Stage B — precision-by-width test (36 predictions)

**Blocked** unless Stage A's gate passed. Add `--force` only as a documented manual
override after reading the Stage A failures.


In [ ]:
!python scripts/run_stage.py --stage stage_b --backend llama-server
!python scripts/score.py --stage stage_b --emit-adjudication
!python scripts/analyze.py --stage stage_b --ni-margin 0.05
!python scripts/package_results.py --stage stage_b


In [ ]:
import json
a = json.load(open('results/stage_b/analysis.json'))
s = a['primary_metric_analysis']
print('scientific outcome :', a['scientific_outcome'])
print('F16-Q4 by width    :', s['f16_q4_gap_by_width'])
print('F16-Q4 overall     :', s['f16_q4_gap_overall'], s['f16_q4_gap_ci'])
print('Q4 verdict         :', s['q4_verdict'])
print('Q8 verdict         :', s['q8_verdict'])


## 8. Stage C — selective end-to-end retrieval (18 predictions)

Six Stage-B questions chosen by a deterministic rule **before** any end-to-end output
exists: 2 low-width, 2 high-width, 2 with the largest fixed-evidence F16/Q4
disagreement. Each document agent now writes its own queries against its private index,
at most two rounds.


In [ ]:
!python scripts/run_stage.py --stage stage_c --backend llama-server
!python scripts/score.py --stage stage_c
!python scripts/analyze.py --stage stage_c --ni-margin 0.05
!python scripts/package_results.py --stage stage_c


## 9. Final report

`RUN_REPORT.md` says plainly whether real llama.cpp inference completed or only the
harness was exercised. Download the zips from `/kaggle/working/results/`.


In [ ]:
from IPython.display import Markdown, display
import pathlib
for stage in ('stage_a','stage_b','stage_c'):
    p = pathlib.Path('results')/stage/'RUN_REPORT.md'
    if p.exists():
        display(Markdown(p.read_text()))
!ls -la results/*.zip 2>/dev/null || echo 'no zips yet'


---
### One-command alternative

```bash
python scripts/run_kaggle.py --stage stage_a
python scripts/run_kaggle.py --stage stage_b   # blocked unless Stage A passed
python scripts/run_kaggle.py --stage stage_c
```

### Harness-only check (no GPU, no model, no science)

```bash
python -m pytest tests -q
SPDQ_ALLOW_FIXTURE=1 python scripts/run_stage.py --stage stage_a --backend fixture
```

That path always ends in `SCIENTIFIC_RECOMMENDATION_NOT_AVAILABLE`. By design.
